# Chạy lại Stanford Dogs: CoOp và DualStream (checkpoint chọn đúng theo validation)

Notebook này sửa lỗi chọn checkpoint của `Semester/train_stanford_dogs_dualstream.py`: sau mỗi `optimizer.step()`, nó **tính lại** text features từ trọng số mới rồi mới chấm validation và lưu đúng context vector tương ứng. CoOp và Gemma-CoOp được train **riêng, tuần tự** để tiết kiệm bộ nhớ; không cần một script chung cho mọi baseline trong paper.

Dữ liệu hiện có là **custom cached split 14.406/6.174**, không phải official Stanford Dogs 12.000/8.580. Notebook chỉ chọn epoch và β bằng validation, sau đó tính test một lần. Kết quả mới lưu trong `notebooks/paper_test/results/`, không ghi đè JSON gốc. Cần `torch`, `transformers` và model `openai/clip-vit-base-patch32` đã cache hoặc có quyền tải. Chạy toàn bộ notebook có thể mất nhiều thời gian trên CPU.

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPTokenizer

def find_repo(start=Path.cwd()):
    for p in (start, *start.parents):
        if (p / "paper/paper_CLIP_Gemma4_revised_v2.md").exists():
            return p
    raise FileNotFoundError("Mở notebook từ trong repository")

ROOT = find_repo()
CACHE = ROOT / "Semester/cached_features/stanford_dogs"
ATTR = ROOT / "Semester/attributes/stanford_dogs_attributes.json"
OUT = ROOT / "notebooks/paper_test/results"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
classes = json.loads((CACHE / "classes.json").read_text(encoding="utf-8"))
attributes = json.loads(ATTR.read_text(encoding="utf-8"))
train_clip = torch.load(CACHE / "train_clip.pt", map_location="cpu", weights_only=True).float()
train_labels = torch.load(CACHE / "train_labels.pt", map_location="cpu", weights_only=True).long()
test_clip = torch.load(CACHE / "test_clip.pt", map_location="cpu", weights_only=True).float()
test_labels = torch.load(CACHE / "test_labels.pt", map_location="cpu", weights_only=True).long()
val_ids, train_ids = [], []
for c in range(len(classes)):
    indices = (train_labels == c).nonzero(as_tuple=True)[0]
    perm = indices[torch.randperm(len(indices))]
    n_val = max(1, int(len(indices)*0.1))
    val_ids.append(perm[:n_val]); train_ids.append(perm[n_val:])
val_idx, train_idx = torch.cat(val_ids), torch.cat(train_ids)
assert len(classes)==120 and len(train_idx)==13024 and len(val_idx)==1382 and len(test_labels)==6174
train_x, train_y = train_clip[train_idx].to(device), train_labels[train_idx].to(device)
val_x, val_y = train_clip[val_idx].to(device), train_labels[val_idx].to(device)
test_x, test_y = test_clip.to(device), test_labels.to(device)
print(device, "train/val/test", len(train_y), len(val_y), len(test_y))

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID)
clip = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
for parameter in clip.parameters():
    parameter.requires_grad_(False)
scale = clip.logit_scale.exp().detach()

templates = {
    "head_and_ears": "head and ears of {name}: {desc}",
    "eyes": "eyes of {name}: {desc}",
    "coat": "coat texture and markings of {name}: {desc}",
    "body": "body structure of {name}: {desc}",
    "distinguishing_mark": "distinguishing trait of {name}: {desc}",
}
concepts = []
with torch.no_grad():
    for c in classes:
        fields = attributes[c]["attributes"]
        texts = [templates[k].format(name=c, desc=fields[k]) for k in templates]
        ids = tokenizer(texts, padding=True, truncation=True, max_length=77, return_tensors="pt").to(device)
        concepts.append(F.normalize(clip.get_text_features(**ids), dim=-1))
W_centroid = F.normalize(torch.stack(concepts).mean(dim=1), dim=-1)
print("Centroid bank:", tuple(W_centroid.shape))

In [ ]:
def make_ids(texts, num_ctx=4):
    rows = []
    for text in texts:
        ids = tokenizer(text, return_tensors="pt")["input_ids"][0]
        body = ids[1:-1][:77-2-num_ctx]
        row = torch.cat([ids[:1], torch.arange(49400, 49400+num_ctx), body, ids[-1:]])
        row = F.pad(row, (0, 77-len(row)))
        rows.append(row)
    return torch.stack(rows).to(device)

base_texts = [str(c) for c in classes]
gemma_texts = [f"{c}, dog, recognized by {attributes[c]['attributes']['distinguishing_mark']}" for c in classes]
ids_base, ids_gemma = make_ids(base_texts), make_ids(gemma_texts)
print("Non-padding token counts (min/median/max):")
for name, ids in [("CoOp",ids_base),("Gemma-CoOp",ids_gemma)]:
    lengths = (ids!=0).sum(dim=1).cpu().numpy()
    print(name, int(lengths.min()), float(np.median(lengths)), int(lengths.max()))
print("Đây là full distinguishing_mark cắt tối đa 77 token; chưa phải quy trình cố định 24 token.")

original_embedding_forward = clip.text_model.embeddings.token_embedding.forward
class SoftPrompt(nn.Module):
    def __init__(self):
        super().__init__()
        self.ctx = nn.Parameter(torch.randn(4, 512, device=device)*0.02)
    def text_features(self, input_ids):
        def replaced(ids, *args, **kwargs):
            embeddings = original_embedding_forward(ids, *args, **kwargs).clone()
            for i in range(4):
                mask = ids == (49400+i)
                if mask.any():
                    embeddings[mask] = self.ctx[i].to(embeddings.dtype)
            return embeddings
        clip.text_model.embeddings.token_embedding.forward = replaced
        try:
            features = clip.get_text_features(input_ids=input_ids)
        finally:
            clip.text_model.embeddings.token_embedding.forward = original_embedding_forward
        return F.normalize(features, dim=-1)

In [ ]:
def top1(logits, y):
    return int((logits.argmax(dim=-1)==y).sum().item()), len(y)

def train_one(ids, name, seed=42, epochs=50, lr=0.005):
    torch.manual_seed(seed)
    model = SoftPrompt().to(device)
    optimizer = torch.optim.AdamW([model.ctx], lr=lr, weight_decay=1e-4)
    best = {"correct":-1, "epoch":None, "ctx":None}
    history = []
    for epoch in range(1, epochs+1):
        model.train()
        text_features = model.text_features(ids)
        loss = F.cross_entropy((train_x @ text_features.T)*scale, train_y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        # Quan trọng: feature validation được tính SAU optimizer.step().
        model.eval()
        with torch.no_grad():
            current_features = model.text_features(ids)
            correct, total = top1((val_x @ current_features.T)*scale, val_y)
            history.append({"epoch":epoch,"val_correct":correct,"val_total":total})
            if correct > best["correct"]:
                best = {"correct":correct,"epoch":epoch,"ctx":model.ctx.detach().clone()}
        if epoch==1 or epoch%10==0 or epoch==epochs:
            print(name, "epoch",epoch,"val",correct,"/",total,"best",best["correct"])
    with torch.no_grad():
        model.ctx.copy_(best["ctx"])
        selected_features = model.text_features(ids).detach()
        selected_correct, _ = top1((val_x @ selected_features.T)*scale, val_y)
    assert selected_correct == best["correct"], "Checkpoint và validation không khớp"
    return selected_features, {"best_epoch":best["epoch"],"val_correct":best["correct"],"val_total":len(val_y),"history":history}

tf_base, base_info = train_one(ids_base, "CoOp")
tf_gemma, gemma_info = train_one(ids_gemma, "Gemma-CoOp")
print("Selected:", base_info["best_epoch"], gemma_info["best_epoch"])

In [ ]:
BETAS = [0.1,0.2,0.35,0.5,0.7,1.0]
with torch.no_grad():
    val_gemma = (val_x @ tf_gemma.T)*scale
    val_cent = (val_x @ W_centroid.T)*scale
    beta_scores = []
    for beta in BETAS:
        n, total = top1(val_gemma+beta*val_cent, val_y)
        beta_scores.append({"beta":beta,"val_correct":n,"val_total":total})
    chosen = max(beta_scores, key=lambda r:r["val_correct"])  # tie -> beta đầu tiên trong BETAS
    beta = chosen["beta"]
    # Test chỉ được tính sau khi epoch và beta đã chọn.
    logits_base = (test_x @ tf_base.T)*scale
    logits_gemma = (test_x @ tf_gemma.T)*scale
    logits_dual = logits_gemma + beta*(test_x @ W_centroid.T)*scale
    predictions = {"coop":logits_base.argmax(-1).cpu(),
                   "gemma_coop":logits_gemma.argmax(-1).cpu(),
                   "dualstream":logits_dual.argmax(-1).cpu()}
result = {"dataset":"Stanford Dogs","split":"custom cached 14,406/6,174; stratified train/val 13,024/1,382",
          "seed":SEED,"model_id":MODEL_ID,"lr":0.005,"epochs":50,
          "checkpoint_selection":"validation feature recomputed after optimizer.step",
          "beta_candidates":BETAS,"beta_val_scores":beta_scores,"selected_beta":beta,
          "base":base_info,"gemma_coop":gemma_info,"test":{}}
for name,pred in predictions.items():
    correct=int((pred==test_labels).sum())
    result["test"][name]={"correct":correct,"total":len(test_labels),"accuracy":100*correct/len(test_labels)}
    print(name, result["test"][name])
print("Selected beta:", beta)

In [ ]:
json_path = OUT / "stanford_dualstream_recheck.json"
pt_path = OUT / "stanford_dualstream_recheck_predictions.pt"
json_path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
torch.save({"train_idx":train_idx,"val_idx":val_idx,"test_labels":test_labels,
            "classes":classes,**{f"pred_{k}":v for k,v in predictions.items()}},pt_path)
print("Saved:",json_path.relative_to(ROOT),pt_path.relative_to(ROOT))
print("Lưu ý: cache không có manifest ID ảnh gốc; kiểm manifest trước khi ghép với run khác để làm McNemar.")